# V4.1 — 01: Setup, EDA, Gate Data


# 00. ACOS Master Pipeline: End-to-End Execution (Production PRO Version)

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction with Implicit Aspects and Opinions**

This production-grade master notebook executes the **entire ACOS benchmark pipeline with 1-Click** on **Google Colab** (with automatic Google Drive persistence at `/content/drive/MyDrive/ACOS`) or on a **Local Environment** (Windows / Linux):

1. **Environment Setup & GPU Diagnostics:** Automatic Google Drive mounting, dependency installation, and detailed GPU / VRAM inspection.
2. **Dynamic Path Architecture (Zero Hardcoded Paths):** Seamless switching between Google Drive `/content/drive/MyDrive/ACOS` and local project workspace.
3. **Pretrained BERT Offline Caching:** Directly caches `bert-base-uncased` from HuggingFace Hub to prevent legacy S3 URL deprecation errors.
4. **Exploratory Data Analysis (EDA):** High-resolution publication plots (300 DPI) and structured CSV exports with auto-caching.
5. **Step 1 (Aspect-Opinion Co-Extraction):** Trains and evaluates BERT-CRF, saves best model checkpoint, tracks peak VRAM, exports `pred4pipeline.txt`, and auto-skips if already trained.
6. **Smart State Checkpoint & Multi-Tier Recovery:** Persists complete runtime state (`pipeline_state.pkl`) and provides seamless fallback auto-loading across all cells.
7. **Candidate Pair Generation Bridge:** Forms Cartesian candidate pairs `(aspect, opinion)` with implicit entity handling `[-1, -1]`.
8. **Step 2 (Category-Sentiment Classification):** Trains multi-label classifier, saves best checkpoint, predicts full quadruples, and auto-skips if already trained.
9. **Benchmark Dashboard & 15 Subtask Metrics:** Evaluates complete quadruple extraction and exports `master_metrics.json`.
10. **MCP (Model Context Protocol) Ready:** Emits real-time `session_manifest.json` tracking pipeline lifecycle states.
11. **Live Interactive Inference:** Color-coded two-stage quadruple extraction on arbitrary customer review text with automatic model loading.

---

### Versi V2 — eksekusi bertahap penuh

Notebook ini adalah turunan dari `00_ACOS_Master_Pipeline_Colab_PRO_Resume.ipynb`.
Perbedaannya: **setiap tahap berat dipecah menjadi sel-sel kecil yang melaporkan
progresnya sendiri**, bukan hanya Step 1.

| Tahap | Sel | Pola |
|---|---|---|
| Pelacak progres | 1b | `step_stage` + `require_vars`, dipakai seluruh notebook |
| Step 1 (BERT-CRF) | 5a-5f | init, cache, data, model, training, laporan |
| Jembatan pasangan | 7a-7b | generate/cache, laporan |
| Step 2 (Category-Sentiment) | 8a-8f | init, cache, data, model, training, laporan |
| Evaluasi final | 9a-9b | evaluasi, tabel & plot |

Setiap sel mencetak judul, langkah bernomor dengan detik berjalan, dan durasi total.
Sel training menulis progres per epoch ke `logs/step*_progress.json` sehingga tetap
terbaca dari Drive bila tab Colab tertutup. Sel yang mahal melewati dirinya sendiri
saat artefaknya sudah ada.

Jalankan sel 1b sekali setelah setiap restart kernel — semua sel tahap
bergantung padanya.

**Metrik mentah**: sel 1b mendefinisikan `patch_eval_metrics_counts()` yang dipanggil di
sel 5a dan 8a, sehingga TP/FP/FN ikut dikembalikan oleh `measureQuad`/`measureQuad_imp`.
Hitungan itu disimpan ke `csv/step*_training_history.csv`, `logs/step*_progress.json`,
`logs/step*_run_result.json`, dan `logs/master_metrics.json`, lalu ditampilkan pada tabel
`master_03`, `master_06`, `master_07`, `master_08`, dan `master_09`.

---

### Versi V4 — IndoBERT fine-tuned + dataset Indonesia (Apps-ACOS)

Turunan dari `00_ACOS_Master_Pipeline_Colab_V2_STAGED.ipynb`, dibangun ulang oleh
`_build_v4_indobert.py` (yang menjalankan generator V2 lebih dulu). Seluruh pola
V2 dipertahankan — sel bertahap `step_stage`, cache per tahap, folder sesi
bertimestamp, tabel `master_*`, dan plot 300 DPI — hanya **backbone** dan
**dataset**-nya yang berganti.

| Aspek | V2 (baseline) | V4 (ini) |
|---|---|---|
| Backbone | `bert-base-uncased` | `indobenchmark/indobert-base-p1`, **fine-tuned di sini** |
| Domain | `rest16` / `laptop` | `appsid` (ulasan aplikasi bank digital) |
| Kategori | 13 (`ENTITAS#ATRIBUT`) | 13 (nama datar, mis. `AUTH_ACCESS`) |
| `num_labels` Step 2 | 39 | **39** (sengaja sama, head tak berubah dimensi) |
| Sumber data | `data/Restaurant-ACOS/` | `data/Apps-ACOS/processed/` → dikonversi di sel 4d |
| Folder sesi | `results/rest16_<ts>/` | `results/appsid_<ts>/` |

Sel baru dibanding V2:

| Sel | Isi | Torch? |
|---|---|---|
| 1s | Sinkronisasi paket `acos_id/` + `sys.path` | tidak |
| 4c | Adapter checkpoint IndoBERT (rekey prefiks `bert.`) + laporan vocab | ya |
| 4d | Gerbang data: taksonomi, split, konversi ACOS, generator `tokenized_data`, gate 2 Inggris | tidak |
| 5d2 | **Gate 1**: bobot encoder dibandingkan numerik dengan checkpoint | ya |

Urutan 4c sebelum 4d disengaja: generator `tokenized_data` memakai vocab
IndoBERT, jadi gerbang data tidak punya tokenizer sebelum adapter selesai.

**Dua kegagalan senyap yang dijaga sel-sel itu.** Pertama, checkpoint IndoBERT
menyimpan key tanpa prefiks `bert.`, sementara loader legacy
(`modeling.py:745`) menetapkan `start_prefix=''` karena `BertForQuadABSA` punya
atribut `self.bert`; tanpa rekey seluruh bobot encoder masuk `missing_keys` dan
logging yang melaporkannya di-comment out (`modeling.py:749-755`) — training
berjalan mulus dengan encoder **acak**. Kedua, `get_labels()` upstream hanya
mengenal `rest*` dan `laptop`; domain lain membuat daftar kategori `None`.
Keduanya diperiksa gate, bukan diasumsikan.

Jalankan sel 1s satu kali setelah setiap restart kernel, sama seperti 1b.

## 1. Environment Setup, Google Drive Mounting & GPU Diagnostics

In [ ]:
# 1. Mount Google Drive jika di Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive berhasil di-mount pada /content/drive")
except Exception:
    print("💻 Berjalan pada lingkungan Lokal / Colab tanpa drive mount.")

# 2. Instalasi dependensi yang dibutuhkan
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3 tqdm


In [ ]:

import os
import sys
import random
import json
import shutil
import pickle
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch


In [ ]:

# 3. GPU Hardware Diagnostics & Optimization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n⚡ Perangkat Komputasi Utama: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cuda_cap = torch.cuda.get_device_capability(0)
    print(f"   GPU Model         : {gpu_name}")
    print(f"   Total VRAM        : {vram_total_gb:.2f} GB")
    print(f"   Compute Capability: {cuda_cap[0]}.{cuda_cap[1]}")
    print(f"   cuDNN Version     : {torch.backends.cudnn.version()}")
    # Aktifkan optimasi benchmark cuDNN untuk matrix/convolution ops
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
    print("   ✅ CUDA Cache dibersihkan & cuDNN benchmark diaktifkan.")
else:
    print("   ℹ️ Berjalan dengan CPU mode.")

### 1b. Pelacak Progres Bertahap (`step_stage`)
Definisi dipakai oleh seluruh sel tahap di bawah. **Wajib dijalankan ulang setiap kali
kernel di-restart**, sebelum melompat ke Step 1/2 atau evaluasi.

Sel ini juga mendefinisikan `patch_eval_metrics_counts()` — patch yang membuat
`measureQuad`/`measureQuad_imp` ikut mengembalikan **TP, FP, FN** di samping
precision/recall/micro-F1, sehingga hitungan mentah bisa disimpan ke CSV/JSON dan
ditampilkan di tabel hasil. Patch-nya dipanggil dari sel 5a dan 8a (setelah
`sys.path` memuat `Extract-Classify-ACOS`), bukan di sini.

In [ ]:
import time


class step_stage:
    """Pelacak progres satu sel: judul, langkah bernomor + waktu, durasi akhir.

    Dipakai seluruh tahap pipeline supaya setiap sel punya jejak sendiri saat
    runtime Colab terputus di tengah eksekusi.
    """

    def __init__(self, title, total_steps=None):
        self.title = title
        self.total = total_steps
        self.n = 0
        self.t0 = None

    def __enter__(self):
        self.t0 = time.time()
        print("=" * 78)
        print(f"▶️  {self.title}")
        print("=" * 78, flush=True)
        return self

    def step(self, msg):
        self.n += 1
        tag = f"{self.n}/{self.total}" if self.total else str(self.n)
        print(f"   [{tag}] {time.time() - self.t0:6.1f}s  {msg}", flush=True)

    def note(self, msg):
        print(f"        {msg}", flush=True)

    def __exit__(self, exc_type, exc, tb):
        dur = time.time() - self.t0
        if exc_type is None:
            print(f"✅ {self.title} — selesai dalam {dur:.1f}s\n", flush=True)
        else:
            print(f"❌ {self.title} — gagal setelah {dur:.1f}s: {exc}\n", flush=True)
        return False


def require_vars(*names):
    """Menghentikan sel dengan pesan jelas bila sel prasyarat belum dijalankan."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Variabel {missing} belum ada di memori. Jalankan sel tahap sebelumnya "
            f"(atau sel pemulihan 6b/6c) sebelum sel ini.")


def write_stage_progress(path, **fields):
    """Menulis jejak progres yang bertahan meski runtime terputus."""
    fields["updated_at"] = datetime.now().isoformat()
    with open(path, "w", encoding="utf-8") as pf:
        json.dump(fields, pf, indent=2)
    return path


# Kolom hitungan mentah dilaporkan apa adanya; hanya kolom laju yang dipersenkan.
METRIC_COUNT_COLS = ("tp", "fp", "fn")
METRIC_RATE_COLS = ("precision", "recall", "micro-F1", "f1")


def patch_eval_metrics_counts():
    """Membuat `measureQuad` & `measureQuad_imp` ikut mengembalikan tp/fp/fn.

    Versi upstream mencetak ketiga hitungan itu lalu membuangnya, sehingga
    `pred_eval`/`pair_eval` hanya meneruskan precision/recall/micro-F1 dan
    notebook tidak pernah bisa menyimpan hitungan mentahnya. Patch ini juga
    memperbaiki dua cacat upstream sekaligus: `measureQuad_imp` melakukan
    `return` di luar loop (hanya slot difficulty terakhir yang terbawa) dan
    melempar KeyError untuk teks prediksi yang tidak ada di `text_type`.
    """
    import eval_metrics as _em

    if getattr(_em, "_ACOS_COUNTS_PATCHED", False):
        return _em

    def _prf(tp, fp, fn):
        p = 0.0 if tp + fp == 0 else 1.0 * tp / (tp + fp)
        r = 0.0 if tp + fn == 0 else 1.0 * tp / (tp + fn)
        f = 0.0 if p + r == 0 else 2 * p * r / (p + r)
        return {"precision": p, "recall": r, "micro-F1": f,
                "tp": float(tp), "fp": float(fp), "fn": float(fn)}

    def measureQuad(pred, gold):
        tp = fp = fn = 0.0
        for text in pred:
            cnt = 0
            if text in gold:
                for pair in pred[text]:
                    if pair in gold[text]:
                        cnt += 1
            tp += cnt
            fp += len(pred[text]) - cnt
            if text in gold:
                fn += len(gold[text]) - cnt
        for text in gold:
            if text not in pred:
                fn += len(gold[text])
        print("tp: {}. fp: {}. fn: {}.".format(tp, fp, fn))
        return _prf(tp, fp, fn)

    def measureQuad_imp(pred, gold, text_type):
        tp = [.0] * 5
        fp = [.0] * 5
        fn = [.0] * 5
        for text in pred:
            for dt in text_type.get(text, [4]):
                cnt = 0
                if text in gold:
                    for pair in pred[text]:
                        if pair in gold[text]:
                            cnt += 1
                tp[dt] += cnt
                fp[dt] += len(pred[text]) - cnt
                if text in gold:
                    fn[dt] += len(gold[text]) - cnt
        for text in gold:
            for dt in text_type.get(text, [4]):
                if text not in pred:
                    fn[dt] += len(gold[text])

        per_dt = []
        for i in range(5):
            print("tp: {}. fp: {}. fn: {}.".format(tp[i], fp[i], fn[i]))
            slot = _prf(tp[i], fp[i], fn[i])
            print(i, ': ', slot)
            per_dt.append(slot)
        # Agregat seluruh slot difficulty. Upstream me-return di luar loop
        # sehingga hanya slot i=4 yang terbawa.
        res = _prf(sum(tp), sum(fp), sum(fn))
        # `pair_eval` mem-format tiap nilai dengan {:.2%}, jadi nilai non-skalar
        # tidak boleh masuk dict ini; rincian per slot disimpan di modul.
        _em.LAST_DIFFICULTY_BREAKDOWN = per_dt
        return res

    _em.measureQuad = measureQuad
    _em.measureQuad_imp = measureQuad_imp
    _em.LAST_DIFFICULTY_BREAKDOWN = []
    _em._ACOS_COUNTS_PATCHED = True
    return _em


def history_display_frame(history, epochs_col="epoch"):
    """Riwayat per epoch → DataFrame siap tabel: hitungan mentah + kolom persen."""
    df = pd.DataFrame(history)
    if df.empty:
        return df
    for c in METRIC_RATE_COLS:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").max() <= 1.0:
            df[c] = (pd.to_numeric(df[c], errors="coerce") * 100).round(2)
    rename = {"tp": "TP", "fp": "FP", "fn": "FN", "precision": "Precision_%",
              "recall": "Recall_%", "micro-F1": "Micro_F1_%"}
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    order = [epochs_col, "loss", "TP", "FP", "FN",
             "Precision_%", "Recall_%", "Micro_F1_%"]
    cols = [c for c in order if c in df.columns]
    return df[cols + [c for c in df.columns if c not in cols]]


def metrics_display_frame(res):
    """Satu dict metrik → tabel dua-jenis: hitungan mentah dan laju dalam persen."""
    rows = []
    for k, v in res.items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            continue
        is_count = k in METRIC_COUNT_COLS
        rows.append({"Metrik": {"tp": "TP", "fp": "FP", "fn": "FN"}.get(k, k),
                     "Jenis": "hitungan" if is_count else "laju",
                     "Nilai": float(v),
                     "Tampil": (f"{float(v):.0f}" if is_count
                                else f"{float(v) * 100:.2f}%")})
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Jenis", ascending=False).reset_index(drop=True)
    return df


def best_epoch_row(history, f1_key="micro-F1"):
    """Baris epoch terbaik dari riwayat, apa pun skala kolom F1 yang tersimpan."""
    df = pd.DataFrame(history)
    if df.empty or f1_key not in df.columns:
        return {}, 0.0, 0
    f1 = pd.to_numeric(df[f1_key], errors="coerce")
    idx = int(f1.idxmax())
    best_f1 = float(f1.max())
    if best_f1 > 1.0:  # riwayat lama menyimpan persen
        best_f1 /= 100.0
    epoch = int(df.loc[idx].get("epoch", idx + 1))
    return df.loc[idx].to_dict(), best_f1, epoch


print("🛠️  step_stage, require_vars, write_stage_progress, patch_eval_metrics_counts, "
      "history_display_frame, metrics_display_frame, best_epoch_row siap dipakai.")

## 2. Dynamic Directory Navigation & Path Initialization (Zero Hardcoded Paths)

In [ ]:
# 1. Deteksi dinamis root direktori proyek (Google Drive vs Colab vs Lokal)
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
HAS_DRIVE = os.path.exists("/content/drive/MyDrive")

# Prioritas pencarian folder ACOS di Google Drive
drive_candidates = [
    "/content/drive/MyDrive/ACOS",
    "/content/drive/MyDrive/ACOS-ASLI",
]
if HAS_DRIVE:
    try:
        for _item in sorted(os.listdir("/content/drive/MyDrive")):
            if "acos" in _item.lower():
                _p = os.path.join("/content/drive/MyDrive", _item)
                if os.path.isdir(_p) and _p not in drive_candidates:
                    drive_candidates.append(_p)
    except Exception:
        pass

base_project_dir = None
if HAS_DRIVE:
    for dc in drive_candidates:
        if os.path.isdir(dc) and (os.path.exists(os.path.join(dc, "Extract-Classify-ACOS")) or os.path.exists(os.path.join(dc, "data"))):
            base_project_dir = dc
            break
    if not base_project_dir:
        base_project_dir = "/content/drive/MyDrive/ACOS"
        os.makedirs(base_project_dir, exist_ok=True)
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Mode Google Drive Terdeteksi: {base_project_dir}")
    print(f"📁 Output Sesi akan disimpan persisten di: {save_dir}")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Mode Colab Ephemeral Aktif: {base_project_dir}")
elif os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Mode Lokal Aktif (Current Dir): {base_project_dir}")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Mode Lokal Aktif (Parent Dir): {base_project_dir}")
else:
    base_project_dir = os.path.abspath("ACOS")
    os.makedirs(base_project_dir, exist_ok=True)
    save_dir = os.path.join(base_project_dir, "Output")
    os.makedirs(save_dir, exist_ok=True)
    print(f"💾 Inisialisasi folder ACOS: {base_project_dir}")

# 2. Auto-clone repositori ACOS jika folder inti belum tersedia
extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
absa5_dir = os.path.join(base_project_dir, "absa5")
if not os.path.exists(extract_dir) or not os.path.exists(absa5_dir):
    print(f"📥 Repositori atau modul absa5 belum lengkap di {base_project_dir}. Menyinkronkan dari GitHub...")
    os.system('git clone https://github.com/haisyamalawwab/ACOS.git /tmp/ACOS_clone')
    os.system(f'cp -r /tmp/ACOS_clone/* "{base_project_dir}/"')
    os.system('rm -rf /tmp/ACOS_clone')
    print("✅ Repositori dan modul absa5 berhasil disinkronkan.")

data_root = os.path.join(base_project_dir, "data")
notebooks_dir = os.path.join(base_project_dir, "notebooks")


In [ ]:

# Impor colab_utils dengan pemilihan salinan yang lengkap & robust
import re
import importlib

# Kontrak: 21 nama yang dipakai notebook. Kalau satu hilang, salinan itu ditolak.
REQUIRED_UTILS = (
    "setup_timestamped_run_dir", "download_bert_pretrained", "analyze_and_plot_eda",
    "plot_training_history", "export_benchmark_tables_and_plots",
    "display_quadruple_dataframe", "df_to_markdown", "export_step_table",
    "MarkdownReport", "SubtaskMetricCapture", "plot_subtask_metrics",
    "features_step1", "features_step2", "pair_examples_from_file",
    "resolve_eval_pair_file", "unpack_model_output",
    "detect_acos_project_root", "inspect_acos_drive_structure",
    "verify_session_save_paths", "find_resumable_session", "auto_find_file",
)

# 1. Pulihkan variabel path bila cell [5] berhenti sebelum barisnya tercapai
if "base_project_dir" not in globals() or not base_project_dir:
    base_project_dir = os.path.abspath(".")
if "save_dir" not in globals() or not save_dir:
    save_dir = os.path.join(base_project_dir, "Output")
if "extract_dir" not in globals() or not extract_dir:
    extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
if "notebooks_dir" not in globals() or not notebooks_dir:
    notebooks_dir = os.path.join(base_project_dir, "notebooks")
if "data_root" not in globals() or not data_root:
    data_root = os.path.join(base_project_dir, "data")


def _utils_missing_symbols(path):
    """Nama publik yang tidak didefinisikan di satu file colab_utils.py.

    Membaca teks file, bukan mengimpornya: pemeriksaan ini harus tetap benar
    sebelum pip install selesai, karena colab_utils sendiri butuh torch.
    """
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        return list(REQUIRED_UTILS)
    with open(path, encoding="utf-8") as fh:
        src = fh.read()
    return [n for n in REQUIRED_UTILS
            if not re.search(r"^(?:def|class)\s+%s\b" % re.escape(n), src, re.M)]


def _prepend_sys_path(p):
    """Paksa p ke posisi terdepan walau sudah ada di urutan yang lebih rendah."""
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)


# 2. Pilih salinan colab_utils.py yang lengkap; notebooks/ diperiksa lebih dulu
_utils_candidates = [notebooks_dir, extract_dir, base_project_dir, os.getcwd()]
_utils_dir = None
_utils_report = []
for _d in _utils_candidates:
    _f = os.path.join(_d, "colab_utils.py")
    _miss = _utils_missing_symbols(_f)
    if os.path.isfile(_f):
        _utils_report.append((_f, _miss))
    if not _miss:
        _utils_dir = _d
        break

# 3. Unduh dari induk hanya kalau semua salinan lokal memang tidak layak
if _utils_dir is None:
    print("⚠️ Tidak ada salinan colab_utils.py yang lengkap. Mengunduh dari GitHub...")
    for _f, _miss in _utils_report:
        print(f"   ✗ {_f} — kurang {len(_miss)} simbol: {', '.join(_miss[:4])}...")
    import urllib.request
    _fresh_dir = os.path.join(base_project_dir, "_acos_utils")
    os.makedirs(_fresh_dir, exist_ok=True)
    _target = os.path.join(_fresh_dir, "colab_utils.py")
    for _url in (
        "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py",
        "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/colab_utils.py",
        "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/Extract-Classify-ACOS/colab_utils.py",
    ):
        try:
            urllib.request.urlretrieve(_url, _target)
        except Exception as _e:
            print(f"   ✗ gagal unduh {_url}: {_e}")
            continue
        if not _utils_missing_symbols(_target):
            _utils_dir = _fresh_dir
            print(f"   ✅ Berhasil: {_url}")
            break
        print(f"   ✗ {_url} juga belum lengkap.")
    if _utils_dir is None:
        raise RuntimeError(
            "colab_utils.py lengkap tidak ditemukan secara lokal maupun dari GitHub. "
            "Salin manual notebooks/colab_utils.py dari repo ke " + base_project_dir)

# 4. Sumber terpilih ditaruh paling depan agar salinan usang tidak membayangi
for _p in [notebooks_dir, extract_dir, base_project_dir, _utils_dir]:
    if os.path.isdir(_p):
        _prepend_sys_path(_p)

# 5. Impor bersih: buang cache lama supaya salinan usang tidak dipakai ulang
sys.modules.pop("colab_utils", None)
try:
    colab_utils = importlib.import_module("colab_utils")
except ModuleNotFoundError as _e:
    _dep = getattr(_e, "name", "") or ""
    if _dep and _dep != "colab_utils":
        raise RuntimeError(
            f"colab_utils butuh paket '{_dep}' yang belum ter-install. Jalankan "
            f"sel instalasi dependensi lebih dulu, lalu ulangi sel ini.") from _e
    raise

_missing_attr = [n for n in REQUIRED_UTILS if not hasattr(colab_utils, n)]
if _missing_attr:
    raise ImportError(
        f"colab_utils di {colab_utils.__file__} tidak punya: {_missing_attr}")
globals().update({n: getattr(colab_utils, n) for n in REQUIRED_UTILS})

print(f"🧩 colab_utils aktif       : {colab_utils.__file__}")
print(f"📂 Base Project Directory : {base_project_dir}")
print(f"📁 Extract & Model Dir     : {extract_dir}")
print(f"📁 Dataset Data Dir        : {data_root}")
print(f"📁 Output & Save Directory : {save_dir}")


### 2c. Dua Root & Paket `acos_id/`

V4 memakai **dua root** yang tidak boleh tertukar:

| Variabel | Isi | Ditulis? |
|---|---|---|
| `indo_root` | `ACOS-IndoBERT/` — dataset, `tokenized_data`, `backbones`, `results` | ya, semuanya |
| `acos_root` | `ACOS-ASLI/` — `Extract-Classify-ACOS/` + data rest16 | **tidak**, baca saja |

Seluruh perbedaan Indonesia ada di paket `acos_id/` di bawah `indo_root`,
**bukan** patch pada `Extract-Classify-ACOS/`. Jalur Inggris karena itu tetap
utuh dan bisa dipakai sebagai kontrol: cukup ganti `DOMAIN` kembali ke `rest16`
di sel 3.

Sel ini berada **setelah** dua sel path di atasnya, bukan sebelum: sel-sel itu
menetapkan `base_project_dir` dan `extract_dir` dari hasil deteksi Drive, dan sel
ini yang menimpanya dengan nilai dua-root yang benar. Kalau urutannya dibalik,
penimpaannya justru yang hilang — tanpa pesan apa pun.

Kelengkapan paket diperiksa per-modul, bukan sekadar `import acos_id`, karena
folder yang tersinkron separuh lolos pemeriksaan paket tetapi gagal beberapa sel
kemudian, jauh dari penyebabnya.

Jalankan sel ini sekali setiap kali kernel di-restart, sama seperti 1b.

In [ ]:
# ============================================================
#  Dua root: indo_root (ditulis) & acos_root (baca saja)
# ============================================================
ACOS_ID_MODULES = ("taxonomy", "build_acos", "tokenize_data", "checkpoint",
                   "selftest", "eda", "upstream")

import importlib

# Repo GitHub yang memuat kedua folder. Dipakai hanya bila ACOS-IndoBERT belum ada.
ACOS_REPO_URL = "https://github.com/haisyamalawwab/ACOS.git"


def _cari_indo_root():
    """Folder ACOS-IndoBERT: satu-satunya penanda adalah subfolder acos_id/."""
    kandidat = []
    if os.path.exists("/content/drive/MyDrive"):
        kandidat += ["/content/drive/MyDrive/ACOS-IndoBERT",
                     "/content/drive/MyDrive/ACOS/ACOS-IndoBERT",
                     "/content/drive/MyDrive/ACOS-ASLI/ACOS-IndoBERT"]
    _base = globals().get("base_project_dir") or os.path.abspath(".")
    kandidat += [os.path.join(_base, "ACOS-IndoBERT"),
                 os.path.abspath("ACOS-IndoBERT"),
                 os.path.abspath(os.path.join("..", "ACOS-IndoBERT")),
                 os.path.abspath(".")]
    for _d in kandidat:
        if os.path.isdir(os.path.join(_d, "acos_id")):
            return _d
    return None


indo_root = _cari_indo_root()
if indo_root is None:
    _target = os.path.join(globals().get("base_project_dir") or os.path.abspath("."),
                           "ACOS-IndoBERT")
    print(f"📥 ACOS-IndoBERT belum ada. Menyinkronkan ke {_target} ...")
    _tmp = "/tmp/ACOS_clone_indo"
    os.system(f"rm -rf {_tmp}")
    os.system(f"git clone --depth 1 {ACOS_REPO_URL} {_tmp}")
    _src = os.path.join(_tmp, "ACOS-IndoBERT")
    if not os.path.isdir(os.path.join(_src, "acos_id")):
        raise RuntimeError(
            f"folder ACOS-IndoBERT/acos_id tidak ada di {ACOS_REPO_URL}. Unggah "
            f"folder ACOS-IndoBERT/ ke Drive secara manual, lalu ulangi sel ini.")
    os.makedirs(_target, exist_ok=True)
    os.system(f'cp -r "{_src}/." "{_target}/"')
    os.system(f"rm -rf {_tmp}")
    indo_root = _target
    print("✅ ACOS-IndoBERT tersinkron.")

_acos_id_dir = os.path.join(indo_root, "acos_id")
_missing = [f"{_m}.py" for _m in ACOS_ID_MODULES
            if not os.path.isfile(os.path.join(_acos_id_dir, f"{_m}.py"))
            or os.path.getsize(os.path.join(_acos_id_dir, f"{_m}.py")) == 0]
if _missing:
    raise RuntimeError(f"acos_id tidak lengkap di {_acos_id_dir}; hilang: {_missing}")

# Pastikan modul acos_id/ memiliki pembaruan terbaru (misal: dual-head di taxonomy.py)
_tax_file = os.path.join(_acos_id_dir, "taxonomy.py")
if os.path.isfile(_tax_file):
    try:
        with open(_tax_file, "r", encoding="utf-8") as _tf:
            if "patch_processor_labels_dualhead" not in _tf.read():
                print("⚠️ acos_id/taxonomy.py di storage adalah versi lama. Menyinkronkan pembaruan dari GitHub...")
                _tmp_up = "/tmp/ACOS_update_indo"
                os.system(f"rm -rf {_tmp_up}")
                os.system(f"git clone --depth 1 {ACOS_REPO_URL} {_tmp_up}")
                _src_up = os.path.join(_tmp_up, "ACOS-IndoBERT", "acos_id")
                if os.path.isdir(_src_up):
                    os.system(f'cp -r "{_src_up}/." "{_acos_id_dir}/"')
                os.system(f"rm -rf {_tmp_up}")
                print("✅ acos_id berhasil diperbarui.")
    except Exception as _e:
        print(f"ℹ️ Verifikasi versi taxonomy dilewati: {_e}")


def _prepend_path(p):
    """Paksa p ke posisi terdepan sys.path walau sudah ada di urutan lebih rendah."""
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)


_prepend_path(indo_root)

for _m in list(sys.modules):
    if _m == "acos_id" or _m.startswith("acos_id."):
        del sys.modules[_m]

acos_id = importlib.import_module("acos_id")
acos_taxonomy = importlib.import_module("acos_id.taxonomy")
acos_selftest = importlib.import_module("acos_id.selftest")
acos_ckpt = importlib.import_module("acos_id.checkpoint")
acos_eda = importlib.import_module("acos_id.eda")
acos_upstream = importlib.import_module("acos_id.upstream")

# Root pipeline Inggris. find_upstream() menuntut keempat berkas kunci ada, jadi
# folder bernama benar tapi kosong ditolak di sini — bukan nanti saat impor
# modeling, di mana pesannya tidak menunjuk penyebabnya.
extract_dir = acos_upstream.ensure_path(
    acos_root=globals().get("base_project_dir"))
acos_root = os.path.dirname(extract_dir)

# Seluruh penyimpanan Indonesia di bawah indo_root; tidak satu pun di bawah
# acos_root, supaya repo pipeline Inggris tetap bersih.
save_dir = indo_root
data_root = os.path.join(indo_root, "data")
tokenized_dir = os.path.join(indo_root, "tokenized_data")
backbones_dir = os.path.join(indo_root, "backbones")
for _d in (data_root, tokenized_dir, backbones_dir,
           os.path.join(indo_root, "results"), os.path.join(indo_root, "build")):
    os.makedirs(_d, exist_ok=True)

print(f"🇮🇩 acos_id v{acos_id.__version__}")
print(f"   indo_root  (tulis) : {indo_root}")
print(f"   acos_root  (baca)  : {acos_root}")
print(f"   extract_dir        : {extract_dir}")
print(f"   data / tokenized   : {data_root} | {tokenized_dir}")
print(f"   Domain Indonesia   : {acos_taxonomy.DOMAIN}")
print(f"   Kategori           : {len(acos_taxonomy.CATEGORIES)} "
      f"→ num_labels Step 2 = {acos_taxonomy.num_labels_step2()}")
print(f"   Label sekuens S1   : {list(acos_taxonomy.SEQ_LABELS)}")
print(f"   Gerbang torch-free : {', '.join(acos_selftest.TORCH_FREE_GATES)}")

## 3. Master Pipeline Parameters, BERT Caching & Session Manifest

In [ ]:
# ============================================================
#  Konfigurasi V4 — IndoBERT fine-tuned + dataset Indonesia
# ============================================================
# Pilihan Domain Dataset:
#   'appsid'  → data/Apps-ACOS   (Indonesia, 13 kategori, backbone IndoBERT)
#   'rest16'  → data/Restaurant-ACOS (Inggris, kontrol; backbone jadi bert-en)
#   'laptop'  → data/Laptop-ACOS     (Inggris, 121 kategori)
DOMAIN = "appsid"

# Backbone. Untuk domain Indonesia dipakai apa adanya; untuk domain Inggris
# dipaksa ke 'bert-en' di bawah, supaya kontrol tetap benar-benar kontrol.
#   'indobert'       → indobenchmark/indobert-base-p1  (target utama)
#   'indobert-large' → indobenchmark/indobert-large-p1 (VRAM T4 mepet)
#   'bert-en'        → bert-base-uncased               (kontrol Inggris)
BACKBONE = "indobert"

# Hyperparameter Pelatihan
MAX_SEQ_LENGTH = 128
STEP1_BATCH_SIZE = 32
STEP2_BATCH_SIZE = 32
STEP1_LR = 2e-5
STEP2_LR = 5e-5
NUM_EPOCHS = 15      # 15 epoch optimal untuk Colab GPU T4/A100 (Default paper: 30)
SEED = 42

# Mode per-epoch: jumlah epoch yang dilatih dalam satu kali eksekusi sel training.
# 0 = jalankan semua epoch sekaligus (perilaku lama).
# 1 = train 1 epoch, simpan state, berhenti — untuk lanjut di sesi Colab berikutnya.
MAX_EPOCHS_THIS_RUN = 1

# Mixed Precision (AMP): akselerasi FP16 pada T4/A100 via tensor core.
# Nonaktifkan (False) hanya jika GradScaler menyebabkan NaN pada optimizer lama.
USE_AMP = True

# Early Stopping: berhenti jika F1 tidak membaik selama N epoch berturut-turut
# (menghemat waktu Colab saat training sudah plateau). Set PATIENCE = 0 untuk
# menonaktifkan early stopping.
PATIENCE = 5
MIN_EPOCHS_BEFORE_STOP = 5  # Training minimal jalan N epoch dulu sebelum early stopping aktif

# do_lower_case WAJIB True untuk indobert-base-p1: tokenizer_config.json-nya
# kosong ({}), jadi tidak ada default yang bisa dipercaya, dan tanpa lowercasing
# token berhuruf kapital berubah menjadi [UNK] dalam jumlah besar.
DO_LOWER_CASE = True

_IS_ID_DOMAIN = str(DOMAIN).lower().startswith("apps")
if not _IS_ID_DOMAIN and BACKBONE != "bert-en":
    print(f"ℹ️ DOMAIN='{DOMAIN}' berbahasa Inggris → BACKBONE dipaksa 'bert-en' "
          f"(semula '{BACKBONE}'). Vocab IndoBERT pada data Inggris menghasilkan "
          f"[UNK] masif dan angkanya tidak bisa dibandingkan.")
    BACKBONE = "bert-en"

# Satu folder cache per backbone. Bukan kenyamanan: kalau IndoBERT dan
# bert-base-uncased berbagi folder, checkpoint yang satu menimpa yang lain dan
# tokenizer tetap memuat vocab yang salah tanpa pesan error — seluruh token
# Indonesia menjadi [UNK] dan yang terlihat hanya F1 rendah.
BACKBONE_DIRNAME = {
    "indobert": "indobert_base_p1",
    "indobert-large": "indobert_large_p1",
    "bert-en": "bert_base_uncased",
}


def _backbone_dirname(backbone=None):
    """Nama folder cache untuk sebuah backbone; dipakai juga sel pemulihan state."""
    key = backbone or globals().get("BACKBONE") or "bert-en"
    return BACKBONE_DIRNAME.get(key, str(key).replace("-", "_"))


# `tokenized_base` adalah argumen `data_dir` yang diberikan ke processor upstream.
# Processor menyusun sendiri `<data_dir>/tokenized_data/<domain>_..._quad_bert.tsv`,
# jadi satu variabel ini yang menentukan Step 1/Step 2 membaca berkas Indonesia di
# indo_root atau berkas Inggris di repo upstream — tanpa menyalin apa pun ke sana.
tokenized_base = indo_root if _IS_ID_DOMAIN else extract_dir
print(f"📚 tokenized_base : {tokenized_base}")

# Reproducibility seeding
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

active_save_dir = indo_root

# Sesi dilanjutkan bila ada artefak tersimpan; set False untuk memaksa sesi baru.
RESUME_LAST_SESSION = True

# 1. Direktori sesi. Seluruhnya di bawah indo_root — repo pipeline Inggris tidak
#    pernah menerima artefak run.
results_base = os.path.join(indo_root, "results")

print(f"🇮🇩 Domain   : {DOMAIN} ({'Indonesia' if _IS_ID_DOMAIN else 'Inggris (kontrol)'})")
print(f"🧠 Backbone : {BACKBONE}")
print(f"📁 Sesi     : {results_base}/{DOMAIN}_<timestamp>/")

In [ ]:

def session_dirs_from_root(run_dir):
    """Menyusun ulang peta direktori sesi dengan kunci yang sama seperti
    setup_timestamped_run_dir(), tanpa membuat folder timestamp baru."""
    dirs = {
        "root": run_dir,
        "checkpoints": os.path.join(run_dir, "checkpoints"),
        "step1_checkpoint": os.path.join(run_dir, "checkpoints", "step1_best"),
        "step2_checkpoint": os.path.join(run_dir, "checkpoints", "step2_best"),
        "plots": os.path.join(run_dir, "plots"),
        "csv": os.path.join(run_dir, "csv"),
        "md": os.path.join(run_dir, "md"),
        "logs": os.path.join(run_dir, "logs"),
    }
    for _p in dirs.values():
        os.makedirs(_p, exist_ok=True)
    return dirs


In [ ]:

def session_cache_score(run_dir):
    """Jumlah artefak kunci di sebuah direktori sesi (0 berarti sesi kosong)."""
    marks = [
        os.path.join(run_dir, "pipeline_state.pkl"),
        os.path.join(run_dir, "csv", "master_01_statistik_dataset.csv"),
        os.path.join(run_dir, "logs", "pred4pipeline.txt"),
        os.path.join(run_dir, "checkpoints", "step1_best", "pytorch_model.bin"),
        os.path.join(run_dir, "checkpoints", "step2_best", "pytorch_model.bin"),
        os.path.join(run_dir, "logs", "master_metrics.json"),
    ]
    return sum(1 for m in marks if os.path.exists(m))


In [ ]:
# Kandidat lokasi pencarian sesi terdahulu. Hanya di bawah indo_root: sesi milik
# pipeline Inggris memakai folder lain dan tidak boleh ikut dipilih, karena
# checkpoint-nya memakai vocab yang berbeda.
candidate_result_roots = [
    results_base,
    os.path.join(indo_root, "Output", "results"),
    "/content/drive/MyDrive/ACOS-IndoBERT/results",
    "/content/drive/MyDrive/ACOS/ACOS-IndoBERT/results",
    "/content/drive/MyDrive/ACOS-ASLI/ACOS-IndoBERT/results",
]

_resume_root = find_resumable_session(candidate_result_roots, DOMAIN) if RESUME_LAST_SESSION else None
if _resume_root:
    session_dirs = session_dirs_from_root(_resume_root)
    print(f"♻️ Melanjutkan sesi tersimpan: {_resume_root}")
    print(f"   Artefak kunci terdeteksi: {session_cache_score(_resume_root)}/6")
else:
    session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)

# Verifikasi integritas dan izin simpan sesi
verify_session_save_paths(session_dirs, domain=DOMAIN)

# 2. Backbone cache di bawah indo_root/backbones, satu folder per backbone
#    (nama dari _backbone_dirname() di sel 3). Isinya diunduh & direkey di sel 4c
#    — di sini hanya path-nya yang ditetapkan.
bert_cache_dir = os.path.join(backbones_dir, _backbone_dirname(BACKBONE))
os.makedirs(bert_cache_dir, exist_ok=True)

if BACKBONE == "bert-en":
    # Jalur kontrol: fungsi V2 apa adanya, tanpa rekey.
    download_bert_pretrained(target_dir=bert_cache_dir)
else:
    _n_ada = sum(1 for _f in ("config.json", "pytorch_model.bin", "vocab.txt")
                 if os.path.exists(os.path.join(bert_cache_dir, _f)))
    print(f"🧠 Backbone dir : {bert_cache_dir} ({_n_ada}/3 berkas ada)")
    print("   Unduh & rekey dilakukan di sel 4c (jangan pakai download_bert_pretrained "
          "untuk IndoBERT — fungsi itu selalu mengunduh bert-base-uncased).")

print(f"\n📁 Active Session Folder: {session_dirs['root']}")
plots_dir = session_dirs["plots"]
csv_dir = session_dirs["csv"]
md_dir = session_dirs["md"]
logs_dir = session_dirs["logs"]

In [ ]:

# Inisialisasi Akumulator Laporan Markdown
rep = MarkdownReport(
    f"00 - Master Pipeline ACOS End-to-End [{DOMAIN.upper()}]",
    md_dir,
    filename="00_master_pipeline.md",
    meta={
        "domain": DOMAIN, "epochs": NUM_EPOCHS,
        "step1_batch": STEP1_BATCH_SIZE, "step2_batch": STEP2_BATCH_SIZE,
        "step1_lr": STEP1_LR, "step2_lr": STEP2_LR,
        "max_seq_length": MAX_SEQ_LENGTH, "seed": SEED,
        "device": str(device), "session_dir": session_dirs["root"],
    },
)


In [ ]:

# Helper Manifest MCP (Model Context Protocol Status Logger)
def update_mcp_manifest(status_str, stage_num, extra_info=None):
    manifest_path = os.path.join(session_dirs["root"], "session_manifest.json")
    manifest_data = {
        "session_id": os.path.basename(session_dirs["root"]),
        "status": status_str,
        "stage": stage_num,
        "domain": DOMAIN,
        "device": str(device),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "hyperparameters": {
            "epochs": NUM_EPOCHS,
            "max_seq_length": MAX_SEQ_LENGTH,
            "step1_batch_size": STEP1_BATCH_SIZE,
            "step2_batch_size": STEP2_BATCH_SIZE,
            "step1_lr": STEP1_LR,
            "step2_lr": STEP2_LR,
            "seed": SEED
        },
        "session_dirs": session_dirs,
        "last_updated": datetime.now().isoformat()
    }
    if extra_info:
        manifest_data.update(extra_info)
    with open(manifest_path, "w", encoding="utf-8") as mf:
        json.dump(manifest_data, mf, indent=2)
    return manifest_path


In [ ]:

# Helper Penyimpanan State Terpadu (Menyimpan seluruh variabel & artefak runtime ke pipeline_state.pkl)
def save_pipeline_state(extra_runtime=None):
    """Menyimpan seluruh konfigurasi, path direktori, dan artefak runtime ke pipeline_state.pkl"""
    state_file = os.path.join(session_dirs["root"], "pipeline_state.pkl")
    completed_stages = []
    _sd = session_dirs
    if os.path.exists(os.path.join(_sd["csv"], "master_01_statistik_dataset.csv")):
        completed_stages.append("EDA")
    if os.path.exists(os.path.join(_sd["step1_checkpoint"], "pytorch_model.bin")) or os.path.exists(os.path.join(_sd["logs"], "pred4pipeline.txt")):
        completed_stages.append("STEP1")
    if os.path.exists(os.path.join(tokenized_base, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")):
        completed_stages.append("PAIRS")
    if os.path.exists(os.path.join(_sd["step2_checkpoint"], "pytorch_model.bin")):
        completed_stages.append("STEP2")
    if os.path.exists(os.path.join(_sd["logs"], "master_metrics.json")):
        completed_stages.append("EVAL")

    _serializable = {}
    for _v in ["label_list_step1", "label_list_step2", "label_map_seq",
               "num_labels_step1", "num_labels_step2", "best_step1_f1",
               "best_step1_epoch", "best_step2_f1", "best_step2_epoch",
               "pakai_1st", "df_pairs"]:
        if _v in globals():
            _serializable[_v] = globals()[_v]
        else:
            _serializable[_v] = None

    if "args_h" in globals() and globals()["args_h"] is not None:
        _serializable["args_h"] = {
            "output_dir": getattr(globals()["args_h"], "output_dir", _sd["logs"]),
            "max_seq_length": getattr(globals()["args_h"], "max_seq_length", MAX_SEQ_LENGTH),
        }

    if extra_runtime:
        _serializable.update(extra_runtime)

    state_data = {
        "DOMAIN": DOMAIN,
        "BACKBONE": globals().get("BACKBONE", "bert-en"),
        "indo_root": globals().get("indo_root", base_project_dir),
        "acos_root": globals().get("acos_root", base_project_dir),
        "tokenized_dir": globals().get("tokenized_dir", ""),
        "base_project_dir": base_project_dir,
        "extract_dir": extract_dir,
        "data_root": data_root,
        "bert_cache_dir": bert_cache_dir,
        "session_dirs": session_dirs,
        "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
        "NUM_EPOCHS": NUM_EPOCHS,
        "STEP1_BATCH_SIZE": STEP1_BATCH_SIZE,
        "STEP2_BATCH_SIZE": STEP2_BATCH_SIZE,
        "STEP1_LR": STEP1_LR,
        "STEP2_LR": STEP2_LR,
        "SEED": SEED,
        "device_str": str(device),
        "completed_stages": completed_stages,
        "runtime": _serializable,
    }

    with open(state_file, "wb") as sf:
        pickle.dump(state_data, sf)

    # Simpan pointer state terbaru untuk domain ini di results_base
    if "results_base" in globals() and os.path.isdir(results_base):
        _pointer_state = os.path.join(results_base, f"latest_pipeline_state_{DOMAIN}.pkl")
        try:
            with open(_pointer_state, "wb") as pf:
                pickle.dump(state_data, pf)
        except Exception:
            pass

    # Simpan JSON label terpisah untuk cadangan
    if globals().get("label_list_step1") is not None:
        with open(os.path.join(_sd["csv"], "labels_step1.json"), "w", encoding="utf-8") as jf:
            json.dump(globals()["label_list_step1"], jf, ensure_ascii=False, indent=2)
    if globals().get("label_list_step2") is not None:
        with open(os.path.join(_sd["csv"], "labels_step2.json"), "w", encoding="utf-8") as jf:
            json.dump(globals()["label_list_step2"], jf, ensure_ascii=False, indent=2)

    return state_file

In [ ]:

# Helper Pencari File di Berbagai Lokasi Sesi (Fallback Search)
def auto_find_file(filename, search_roots=None, must_contain=None, domain=None, min_size_bytes=0):
    """Mencari berkas di direktori sesi aktif atau sesi terdahulu dengan filter:
    - must_contain: memastikan path memuat substring tertentu (mis. 'step1_best')
    - domain: memastikan tidak mengambil file dari domain lain (mis. laptop saat domain=rest16)
    - min_size_bytes: memastikan file tidak kosong / rusak"""
    if domain is None and "DOMAIN" in globals():
        domain = globals()["DOMAIN"]
    if search_roots is None:
        search_roots = [
            session_dirs.get("root", ""),
            results_base if 'results_base' in globals() else "",
            "/content/drive/MyDrive/ACOS/Output/results",
            "/content/drive/MyDrive/ACOS/results",
            "/content/drive/MyDrive/ACOS-ASLI/Output/results",
            "/content/drive/MyDrive/ACOS-ASLI/results",
            os.path.join(base_project_dir, "Output", "results") if 'base_project_dir' in globals() else "",
            os.path.join(base_project_dir, "results") if 'base_project_dir' in globals() else "",
        ]
    elif isinstance(search_roots, str):
        search_roots = [search_roots]

    for sr in search_roots:
        if not sr or not os.path.exists(sr):
            continue
        for root, dirs, files in os.walk(sr):
            if filename in files:
                hit = os.path.join(root, filename)
                norm = hit.replace(os.sep, "/")
                if must_contain and must_contain not in norm:
                    continue
                if domain and f"/{domain}_" not in norm and f"_{domain}/" not in norm and f"/{domain}/" not in norm:
                    other_domains = ["laptop", "rest16"]
                    if any(f"/{od}_" in norm for od in other_domains if od != domain):
                        continue
                if min_size_bytes > 0:
                    try:
                        if os.path.getsize(hit) < min_size_bytes:
                            continue
                    except Exception:
                        continue
                return hit
    return None

m_path = update_mcp_manifest("INITIALIZED", 1)
print(f"📡 MCP Session Manifest diinisialisasi: {m_path}")

# Konfigurasi Tabel
df_cfg = pd.DataFrame([
    {"Parameter": "domain", "Nilai": DOMAIN},
    {"Parameter": "num_epochs", "Nilai": NUM_EPOCHS},
    {"Parameter": "step1_batch_size", "Nilai": STEP1_BATCH_SIZE},
    {"Parameter": "step2_batch_size", "Nilai": STEP2_BATCH_SIZE},
    {"Parameter": "step1_learning_rate", "Nilai": STEP1_LR},
    {"Parameter": "step2_learning_rate", "Nilai": STEP2_LR},
    {"Parameter": "max_seq_length", "Nilai": MAX_SEQ_LENGTH},
    {"Parameter": "seed", "Nilai": SEED},
    {"Parameter": "device", "Nilai": str(device)},
])
rep.section("1. Konfigurasi pipeline")
export_step_table(df_cfg, name="master_00_konfigurasi", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Konfigurasi Master Pipeline ({DOMAIN.upper()})")
rep.table(df_cfg, caption="Hyperparameter")
save_pipeline_state()
print("💾 Inisialisasi state awal tersimpan.")

## 4. Exploratory Data Analysis (EDA) & Publication Visualizations

In [ ]:
# Eksekusi Analisis Data Eksploratif (dengan Pemeriksaan Cache Otomatis)
eda_stats_csv = os.path.join(csv_dir, "master_01_statistik_dataset.csv")
eda_ringkas_csv = os.path.join(csv_dir, "master_02_ringkasan_eda.csv")
# Plot wajib ikut ada. Tanpa syarat ini jalur cache melewati pembuatan
# gambar dan bagian laporan EDA berakhir tanpa visualisasi.
eda_plot_utama = os.path.join(plots_dir, "01_eda_dataset_distribution.png")


In [ ]:

if 'df_stats' in globals() and df_stats is not None and not df_stats.empty and 'df_records' in globals() and df_records is not None:
    print("⏩ [CACHE HIT] Menggunakan objek df_stats & df_records dari memori runtime.")
elif (os.path.exists(eda_stats_csv) and os.path.exists(eda_ringkas_csv)
      and os.path.exists(eda_plot_utama)):
    print(f"⏩ [CACHE HIT] Memuat hasil EDA tersimpan dari: {eda_stats_csv}")
    df_stats = pd.read_csv(eda_stats_csv)
    df_ringkas = pd.read_csv(eda_ringkas_csv)
    df_records = pd.DataFrame()
elif acos_taxonomy.is_id_domain(DOMAIN):
    # colab_utils.analyze_and_plot_eda() memetakan domain lewat tabel tertutup
    # {rest16, laptop} dengan fallback ke Restaurant-ACOS — domain 'appsid' TIDAK
    # error di sana, ia diam-diam melaporkan statistik dataset Inggris. Karena itu
    # jalur Indonesia memakai fungsi sendiri dengan kontrak keluaran identik.
    print("📊 Menjalankan EDA dataset Indonesia (acos_id.eda)...")
    df_stats, df_records = acos_eda.analyze_and_plot_eda_id(
        data_dir=data_root,
        domain=DOMAIN,
        output_plots_dir=plots_dir,
        output_csv_dir=csv_dir,
    )
else:
    # Kontrol Inggris: data rest16/laptop dibaca dari repo ACOS-ASLI.
    print("📊 Menjalankan Analisis Data Eksploratif (EDA)...")
    df_stats, df_records = analyze_and_plot_eda(
        data_dir=acos_root,
        domain=DOMAIN,
        output_plots_dir=plots_dir,
        output_csv_dir=csv_dir,
    )

In [ ]:

rep.section("2. Eksplorasi data")
if df_stats is None or df_stats.empty:
    print("⚠️ [Peringatan] Statistik EDA kosong: pastikan folder data/ tersedia.")
    rep.text("Statistik EDA kosong; folder `data/` tidak ditemukan.")
else:
    export_step_table(df_stats, name="master_01_statistik_dataset", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Statistik Dataset {DOMAIN.upper()} per Split")
    rep.table(df_stats, caption="Statistik per split")

    if 'df_records' in globals() and df_records is not None and not df_records.empty:
        n = len(df_records)
        df_ringkas = pd.DataFrame([{
            "Total_Quadruple": n,
            "Implicit_Aspect": int(df_records["Is_Implicit_Aspect"].sum()),
            "Implicit_Opinion": int(df_records["Is_Implicit_Opinion"].sum()),
            "Keduanya_Implicit": int((df_records["Is_Implicit_Aspect"] & df_records["Is_Implicit_Opinion"]).sum()),
            "Kategori_Unik": int(df_records["Category"].nunique()),
            "Panjang_Kalimat_Median": float(df_records["Text_Length"].median()),
        }])
        export_step_table(df_ringkas, name="master_02_ringkasan_eda", csv_dir=csv_dir, md_dir=md_dir,
                          title=f"Ringkasan EDA ({DOMAIN.upper()})")
        rep.table(df_ringkas, caption="Ringkasan EDA")

    from IPython.display import Image, display
    for fname, cap in [
        ("01_eda_dataset_distribution.png", "Komposisi dataset & explicit vs implicit"),
        ("02_eda_category_sentiment.png", "Kategori teratas & polaritas sentimen"),
        ("02b_eda_length_and_implicit_combo.png", "Panjang kalimat & kombinasi implicit"),
        ("02c_eda_category_sentiment_heatmap.png", "Heatmap kategori x sentimen"),
    ]:
        p = os.path.join(plots_dir, fname)
        if os.path.exists(p):
            display(Image(p))
            rep.image(p, cap)


In [ ]:

tot_q = int(df_ringkas["Total_Quadruple"].values[0]) if ('df_ringkas' in globals() and not df_ringkas.empty) else 0
update_mcp_manifest("EDA_COMPLETED", 2, {"total_quadruples": tot_q})
save_pipeline_state()
print("✅ EDA selesai dan status disimpan ke pipeline_state.pkl.")

### 4b. Diagnostik Lokasi Dataset & Tokenized Data

In [ ]:
# Diagnostik Struktur Folder Drive, Dataset, BERT Cache & Sesi Output
drive_audit_report = inspect_acos_drive_structure(base_project_dir=base_project_dir, domain=DOMAIN, verbose=True)


### 4c. Adapter Checkpoint IndoBERT

Menyiapkan `bert_cache_dir` berisi `config.json`, `pytorch_model.bin`, dan
`vocab.txt` IndoBERT, dengan state_dict yang sudah **direkey**: setiap key
diberi prefiks `bert.` agar cocok dengan `start_prefix=''` yang dipakai loader
legacy pada kelas yang punya atribut `self.bert`.

Sel ini harus berjalan **sebelum** gerbang data 4d, bukan sesudahnya: generator
`tokenized_data` memakai vocab IndoBERT, jadi gate `tokenized` tidak punya
tokenizer sebelum sel ini selesai.

Idempoten lewat penanda `_rekey.json` di folder yang sama. Penanda itu bukan
kenyamanan: menjalankan rekey dua kali menghasilkan `bert.bert.embeddings...`,
dan hasilnya sama buruknya dengan tidak merekey sama sekali.

Laporan vocab akan menunjukkan `config.vocab_size = 50000` sementara `vocab.txt`
berisi 30.521 token. Itu memang begitu untuk `indobert-base-p1` — matriks
embedding-nya benar-benar 50.000 baris dan id 30.521+ tidak pernah terpakai.
Jangan pernah memakai `config.vocab_size` sebagai jumlah token.

In [ ]:
require_vars("step_stage", "acos_ckpt", "bert_cache_dir", "BACKBONE")

with step_stage("4c. Adapter checkpoint IndoBERT (rekey prefiks bert.)", 5) as st:
    if not acos_taxonomy.is_id_domain(DOMAIN):
        st.step(f"DOMAIN='{DOMAIN}' — memakai bert-base-uncased apa adanya, "
                f"adapter dilewati")
        backbone_report = {"dilewati": True}
    else:
        st.step(f"Backbone: {BACKBONE} → {acos_ckpt.BACKBONES[BACKBONE]['hf_id']}")
        st.step(f"Target   : {bert_cache_dir}")
        backbone_report = acos_ckpt.prepare_backbone(BACKBONE, bert_cache_dir)

        _rk = backbone_report["rekey"]
        if _rk.get("dilewati"):
            st.step(f"Rekey dilewati — {_rk['dilewati']}")
        else:
            st.step(f"Rekey: {_rk['n_diberi_prefiks']} dari {_rk['n_key']} key "
                    f"diberi prefiks 'bert.'")
        st.note(f"key sebelum: {_rk.get('key_sebelum')}")
        st.note(f"key sesudah: {_rk.get('key_sesudah')}")

        _v = backbone_report["vocab"]
        st.step(f"config.vocab_size={_v.get('config_vocab_size')} | "
                f"vocab.txt={_v.get('vocab_lines')} token | "
                f"hidden={_v.get('hidden_size')} × {_v.get('num_hidden_layers')} layer")
        if not _v.get("konsisten", True):
            st.note(f"⚠️ selisih {_v.get('selisih')} — normal untuk indobert-base-p1; "
                    f"rujuk vocab.txt, JANGAN config.vocab_size")

        df_backbone = pd.DataFrame([{
            "Backbone": BACKBONE,
            "HF_ID": acos_ckpt.BACKBONES[BACKBONE]["hf_id"],
            "Key_Total": _rk.get("n_key"),
            "Key_Diberi_Prefiks": _rk.get("n_diberi_prefiks"),
            "Key_Berprefiks_bert": _rk.get("n_key_berprefiks_bert"),
            "Config_Vocab_Size": _v.get("config_vocab_size"),
            "Vocab_Txt_Token": _v.get("vocab_lines"),
            "Hidden_Size": _v.get("hidden_size"),
            "Layer": _v.get("num_hidden_layers"),
        }])
        export_step_table(df_backbone, name="master_00_backbone_indobert",
                          csv_dir=csv_dir, md_dir=md_dir,
                          title="Adapter Checkpoint IndoBERT")
        rep.section("1b. Backbone & gerbang data")
        rep.table(df_backbone, caption="Backbone & hasil rekey")

        with open(os.path.join(session_dirs["logs"], "backbone_report.json"),
                  "w", encoding="utf-8") as jf:
            json.dump(backbone_report, jf, indent=2, ensure_ascii=False, default=str)
        st.step("backbone_report.json tersimpan; Gate 1 numerik menyusul di sel 5d2")

### 4d. Gerbang Data Indonesia (wajib sebelum training)

Lima gerbang, semuanya torch-free, dijalankan berurutan. Kalau ada yang merah sel
ini melempar exception alih-alih melanjutkan — setiap kegagalan di sini tidak
terlihat dari kurva loss maupun metrik training.

| Gate | Yang diperiksa | Kalau gagal |
|---|---|---|
| `taxonomy` | 13 kategori di kode == `label_maps.json`, **urutan sama** | indeks head Step 2 bergeser, seluruh prediksi kategori salah |
| `dataset` | berkas sumber ada; `review_id` train/dev/test saling lepas | kebocoran data, angka test terlalu tinggi |
| `acos_build` | `appsid_quad_*.tsv` terbentuk; tiap span menunjuk token nyata | span rusak jadi label `O`, aspek hilang tanpa pesan |
| `tokenized` | retokenisasi WordPiece tidak menghilangkan satu tuple pun | data training menyusut senyap |
| `gate2_english` | regenerasi data Inggris **identik** dengan `tokenized_data/` di repo | konvensi offset generator salah; ini satu-satunya bukti eksternalnya |

`tokenized` memakai vocab IndoBERT dari sel 4c, jadi 4c harus sudah selesai.

`gate2_english` memberi toleransi satu kalimat pada `rest16_train_pair.tsv`.
Itu cacat data upstream: `rest16_quad_train.tsv` baris 451 memuat span opini
lebar-nol `3,3`, dan berkas repo memetakan baris itu tidak konsisten antara
`*_quad_bert.tsv` (`3,4`) dan `*_pair.tsv` (`3,5`, plus satu pasangan hilang).
Generator mengikuti berkas quad, yang dipakai Step 1.

In [ ]:
require_vars("step_stage", "acos_selftest", "acos_taxonomy", "DOMAIN")

# Set True untuk membangun ulang berkas ACOS & tokenized_data dari nol.
FORCE_REBUILD_ID_DATA = False

with step_stage("4d. Gerbang data Indonesia (5 gate torch-free)", 6) as st:
    if not acos_taxonomy.is_id_domain(DOMAIN):
        st.step(f"DOMAIN='{DOMAIN}' bukan domain Indonesia — seluruh gate dilewati")
        st.note("Notebook berjalan sebagai kontrol Inggris; sel 4c & 5d2 juga "
                "melewati dirinya sendiri.")
        id_gate_results = {}
    else:
        if not os.path.exists(os.path.join(bert_cache_dir, "vocab.txt")):
            raise RuntimeError(
                f"vocab.txt belum ada di {bert_cache_dir}. Jalankan sel 4c "
                f"(adapter checkpoint IndoBERT) lebih dulu — generator "
                f"tokenized_data memakai vocab itu.")

        # default_paths() menurunkan seluruh path dari dua root; yang ditimpa di
        # sini hanya bert_cache_dir (nama folder tergantung BACKBONE) dan
        # work_dir (keluaran gate 2 disimpan di folder sesi, bukan build/).
        _paths = acos_selftest.default_paths(indo_root, acos_root)
        _paths["bert_cache_dir"] = bert_cache_dir
        _paths["work_dir"] = os.path.join(session_dirs["logs"], "gates")
        st.step(f"Sumber data   : {_paths['data_root']}")
        st.step(f"Vocab IndoBERT: {_paths['bert_cache_dir']}")
        st.note(f"tokenized_data → {_paths['tokenized_dir']}")
        st.note(f"upstream (baca) → {_paths['extract_dir']}")

        # Gate 2 membandingkan dengan berkas Inggris di repo, jadi butuh vocab
        # bert-base-uncased. Diunduh di sini kalau belum ada; ukurannya 232 KB
        # (hanya vocab.txt, bukan pytorch_model.bin).
        if not os.path.exists(os.path.join(_paths["en_vocab_dir"], "vocab.txt")):
            os.makedirs(_paths["en_vocab_dir"], exist_ok=True)
            import urllib.request
            urllib.request.urlretrieve(
                "https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt",
                os.path.join(_paths["en_vocab_dir"], "vocab.txt"))
            st.note("vocab.txt bert-base-uncased diunduh untuk gate 2")

        id_gate_results = acos_selftest.run_gates(
            paths=_paths, only=acos_selftest.TORCH_FREE_GATES,
            rebuild=FORCE_REBUILD_ID_DATA, raise_on_fail=True, verbose=True)
        st.step(f"{len(id_gate_results)} gate dijalankan, semuanya hijau")

        _b = id_gate_results["acos_build"]["detail"]["per_split"]
        _t = id_gate_results["tokenized"]["detail"]["per_split"]
        df_id_gates = pd.DataFrame([
            {"Split": s,
             "Baris_ACOS": _b[s].get("baris", 0),
             "Quad_ACOS": _b[s].get("quad", 0),
             "Quad_Tokenized": _t[s].get("quad", 0),
             "Quad_Hilang": _t[s].get("quad_hilang", 0),
             "Aspek_Eksplisit": _t[s].get("aspek_eksplisit", 0),
             "Aspek_Implisit": _t[s].get("aspek_implisit", 0),
             "Opini_Eksplisit": _t[s].get("opini_eksplisit", 0),
             "Opini_Implisit": _t[s].get("opini_implisit", 0)}
            for s in ("train", "dev", "test") if s in _b and s in _t])
        export_step_table(df_id_gates, name="master_00b_gerbang_data_id",
                          csv_dir=csv_dir, md_dir=md_dir,
                          title="Gerbang Data Indonesia (Apps-ACOS)")
        rep.table(df_id_gates, caption="Konversi & retokenisasi per split")
        st.step("Tabel master_00b ditulis")

        _gate_json = os.path.join(session_dirs["logs"], "id_gates.json")
        with open(_gate_json, "w", encoding="utf-8") as jf:
            json.dump({k: {"ok": v["ok"], "detail": v["detail"]}
                       for k, v in id_gate_results.items()},
                      jf, indent=2, ensure_ascii=False, default=str)
        st.step(f"Hasil gate → {_gate_json}")
        update_mcp_manifest("ID_DATA_GATES_PASSED", 1,
                            {"n_gate": len(id_gate_results),
                             "num_labels_step2": acos_taxonomy.num_labels_step2()})